In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [ ]:
df = pd.read_csv("data/medical_cost_prediction_dataset.csv")

print("Original Shape:", df.shape)

data_limit = 5000

df = df.head(data_limit)

print("After Limit:", df.shape)

df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/medical_cost_prediction_dataset.csv'

In [ ]:
df.info()

df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   age                      5000 non-null   int64  
 1   gender                   5000 non-null   object 
 2   bmi                      5000 non-null   float64
 3   smoker                   5000 non-null   object 
 4   diabetes                 5000 non-null   int64  
 5   hypertension             5000 non-null   int64  
 6   heart_disease            5000 non-null   int64  
 7   asthma                   5000 non-null   int64  
 8   physical_activity_level  5000 non-null   object 
 9   daily_steps              5000 non-null   int64  
 10  sleep_hours              5000 non-null   float64
 11  stress_level             5000 non-null   int64  
 12  doctor_visits_per_year   5000 non-null   int64  
 13  hospital_admissions      5000 non-null   int64  
 14  medication_count        

,age,bmi,diabetes,hypertension,heart_disease,asthma,daily_steps,sleep_hours,stress_level,doctor_visits_per_year,hospital_admissions,medication_count,insurance_coverage_pct,previous_year_cost,annual_medical_cost
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.00000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,53.299000,25.970820,0.207600,0.288000,0.14220,0.096400,7993.216800,6.488140,5.475400,4.030600,1.001000,3.509000,57.953000,10248.515400,8048.886894
std,20.646851,5.046651,0.405629,0.452876,0.34929,0.295169,4052.127069,1.443361,2.892312,2.010689,0.978566,2.292721,31.627742,5626.095015,7071.020228
min,18.000000,6.400000,0.000000,0.000000,0.00000,0.000000,1004.000000,4.000000,1.000000,0.000000,0.000000,0.000000,0.000000,500.000000,404.950000
25%,36.000000,22.600000,0.000000,0.000000,0.00000,0.000000,4545.000000,5.200000,3.000000,3.000000,0.000000,1.000000,53.000000,5342.750000,3241.977500
50%,53.000000,25.900000,0.000000,0.000000,0.00000,0.000000,7989.000000,6.500000,5.000000,4.000000,1.000000,3.000000,71.000000,10281.000000,5539.780000
75%,71.000000,29.400000,0.000000,1.000000,0.00000,0.000000,11532.250000,7.700000,8.000000,5.000000,2.000000,6.000000,79.000000,15034.500000,10094.097500
max,89.000000,43.600000,1.000000,1.000000,1.00000,1.000000,14999.000000,9.000000,10.000000,14.000000,6.000000,7.000000,94.000000,19996.000000,44792.100000


In [ ]:
df = pd.get_dummies(df, drop_first=True)

df.head()

,age,bmi,diabetes,hypertension,heart_disease,asthma,daily_steps,sleep_hours,stress_level,doctor_visits_per_year,...,insurance_coverage_pct,previous_year_cost,annual_medical_cost,gender_Male,smoker_Yes,physical_activity_level_Low,physical_activity_level_Medium,insurance_type_Private,city_type_Semi-Urban,city_type_Urban
0,69,29.4,1,0,0,0,14825,4.4,8,1,...,80,10885,2645.50,True,False,False,True,True,True,False
1,32,22.9,1,0,0,0,3620,6.0,7,4,...,64,18722,10959.70,False,False,False,True,False,True,False
2,89,25.7,0,0,0,0,10578,4.5,7,2,...,0,4196,8409.80,True,False,False,False,False,False,True
3,78,31.9,0,1,0,0,6226,8.6,9,6,...,70,11128,7996.62,True,True,True,False,False,False,True
4,38,27.7,0,0,0,0,6253,5.7,3,6,...,77,15110,3202.52,True,False,False,False,True,False,True


In [ ]:
X = df.drop(["annual_medical_cost", "previous_year_cost"], axis=1)

y = df["annual_medical_cost"]

In [ ]:
splits = {
    "70:30": 0.30,
    "80:20": 0.20,
    "90:10": 0.10
}

In [ ]:
results = []

for split_name, test_size in splits.items():

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=42
    )

    # ---------------- XGBoost ----------------
    xgb_model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        random_state=42
    )

    xgb_model.fit(X_train, y_train)

    y_pred_xgb = xgb_model.predict(X_test)

    results.append({
        "Split": split_name,
        "Model": "XGBoost",
        "MAE": mean_absolute_error(y_test, y_pred_xgb),
        "MSE": mean_squared_error(y_test, y_pred_xgb),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_xgb)),
        "R2 Score": r2_score(y_test, y_pred_xgb)
    })


    # ---------------- LightGBM ----------------
    lgb_model = LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        random_state=42
    )

    lgb_model.fit(X_train, y_train)

    y_pred_lgb = lgb_model.predict(X_test)

    results.append({
        "Split": split_name,
        "Model": "LightGBM",
        "MAE": mean_absolute_error(y_test, y_pred_lgb),
        "MSE": mean_squared_error(y_test, y_pred_lgb),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_lgb)),
        "R2 Score": r2_score(y_test, y_pred_lgb)
    })


    # ---------------- CatBoost ----------------
    cat_model = CatBoostRegressor(
        iterations=200,
        learning_rate=0.1,
        depth=4,
        verbose=0,
        random_state=42
    )

    cat_model.fit(X_train, y_train)

    y_pred_cat = cat_model.predict(X_test)

    results.append({
        "Split": split_name,
        "Model": "CatBoost",
        "MAE": mean_absolute_error(y_test, y_pred_cat),
        "MSE": mean_squared_error(y_test, y_pred_cat),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_cat)),
        "R2 Score": r2_score(y_test, y_pred_cat)
    })


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000815 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 731
[LightGBM] [Info] Number of data points in the train set: 3500, number of used features: 20
[LightGBM] [Info] Start training from score 8016.468199
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000468 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 732
[LightGBM] [Info] Number of data points in the train set: 4000, number of used features: 20
[LightGBM] [Info] Start training from score 8057.088835
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000489 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not eno

In [ ]:
comparison = pd.DataFrame(results)

comparison

,Split,Model,MAE,MSE,RMSE,R2 Score
0,70:30,XGBoost,738.471679,1.412329e+06,1188.414602,0.971990
1,70:30,LightGBM,721.203916,1.233988e+06,1110.850157,0.975527
2,70:30,CatBoost,679.804493,9.805762e+05,990.240482,0.980553
3,80:20,XGBoost,713.222952,1.243020e+06,1114.908192,0.974516
4,80:20,LightGBM,693.364090,1.115671e+06,1056.253514,0.977127
5,80:20,CatBoost,668.161285,9.462980e+05,972.778503,0.980599
6,90:10,XGBoost,708.668535,1.209287e+06,1099.675884,0.974983
7,90:10,LightGBM,693.837487,1.110008e+06,1053.569323,0.977037
8,90:10,CatBoost,657.843888,9.052466e+05,951.444501,0.981273
